In [3]:
import os
from langchain_cerebras import ChatCerebras
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
from pymongo import MongoClient
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException
from langchain.agents import create_agent

load_dotenv()

CEREBRAS_API_KEY=os.getenv("CEREBRAS_API_KEY")
BREVO_API_KEY=os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")


In [4]:
DB_NAME = "soulsync"
COLLECTION_NAME="users"
SENDER_NAME="SoulSync"
SENDER_MAIL="lokeshvijayraina@gmail.com"


In [5]:
llm_gpt=ChatCerebras(model="gpt-oss-120b",api_key=CEREBRAS_API_KEY)
client= MongoClient(MONGODB_URI)
users_collection=client[DB_NAME][COLLECTION_NAME]


In [6]:
class EmailTemplate(BaseModel):
    subject_template:str
    body_template:str
    reason:str

class EmailDraft(BaseModel):
    receiver_mail:str
    subject:str
    body:str

class DispatchedUsers(BaseModel):
    totalUsers:str
    sent:str
    failed:str

In [ ]:
@tool
def find_users(
    filters: dict = None,
    sort_by: str = None,
    ascending: bool = False,
    limit: int = 5,
):
    """
    Query SoulSync users with optional filters, sorting, and limit.

    Available fields:
    - name
    - email
    - totalListeningTime
    - updatedAt (last active)
    - createdAt (joined date)
    """

    query = filters or {}

    targeted_users = users_collection.find(
        query,
        {
            "_id": 0,
            "name": 1,
            "email": 1,
            "totalListeningTime": 1,
            "updatedAt": 1,
            "createdAt": 1,
        },
    )

    if sort_by:
        targeted_users = targeted_users.sort(
            sort_by,
            -1 if not ascending else 1,
        )

    targeted_users = targeted_users.limit(limit)

    return list(targeted_users)

In [14]:
#find_users.invoke({"filters": {}, "sort_by": "totalListeningTime", "ascending": False, "limit": 5})


In [15]:
finder_agent=create_agent(
    model=llm_gpt,
    tools=[find_users],
    system_prompt=
    """
    You are LookOut's user discovery agent.

Your job is to identify the correct SoulSync users by calling the `find_users` tool.

Available fields:

* `name`
* `email`
* `country`
* `totalListeningTime` (seconds)
* `createdAt` (join date)
* `updatedAt` (last active)

Infer these arguments from the user's request:

* `filters`
* `sort_by`
* `ascending`
* `limit`

Intent examples:

* Top / most active → `sort_by="totalListeningTime"`, `ascending=False`
* Least active → `sort_by="totalListeningTime"`, `ascending=True`
* Newest → `sort_by="createdAt"`, `ascending=False`
* Oldest → `sort_by="createdAt"`, `ascending=True`
* Recently active → `sort_by="updatedAt"`, `ascending=False`
* Inactive → `sort_by="updatedAt"`, `ascending=True`
* Country-specific requests → use `filters`

Rules:

* Always call `find_users`.
* Never invent users or fields.
* Combine filters and sorting when needed.
* If the request cannot be answered using the available fields, explain why.

    """)

In [ ]:
response = finder_agent.invoke({
    "messages":[ HumanMessage(content="tell the latest active users today ")]
})
print(response)